# Train the Hangman BiGRU + Attention on Kaggle GPU

Clones the `approach/gru-attention` branch of the project repo and runs
training there. Mirrors `approach/bilstm-attention`'s change on top of
`approach/gru`: a self-attention layer between the recurrent output and
the classification head, so every position gets a direct, weighted view
of every other position instead of only what survives the recurrence.

**Before running:** in the notebook's Settings panel (right sidebar), set
**Accelerator = GPU T4 x2** (or any GPU) and **Internet = On** (needed to `git clone`).

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected -- check Settings > Accelerator in the sidebar')

In [ ]:
REPO_URL = "https://github.com/Sahoo-Achyutananda/MELTWATER---HACKATHON.git"
BRANCH = "approach/gru-attention"

!rm -rf repo
!git clone --branch $BRANCH --single-branch $REPO_URL repo
%cd repo/brand-buzzword-hackathon
!ls

## Train

Same masked-language-model objective as the other branches: randomly mask
letters in each training word, predict the true letter at each masked
position from bidirectional context. At inference: feed the real board
mask through the model, sum per-position letter probabilities across all
blanks, guess the highest-scoring unguessed letter.

In [ ]:
!python src/train_gru.py --epochs 20

## Validate

Same methodology as every other branch, for a fair comparison: hold out
10% of train.txt, play full interactive games against words the model
never trained on. Compare against plain GRU and BiLSTM+attention (47.7%
baseline) to see which combination actually wins.

In [ ]:
!python src/validate_gru.py

## Generate submission.csv

Plays the actual game against every word in test.txt using the model
still in this session. Sandbox leaderboard checkpoint only -- per the
competition's Final Judgement policy, final hiring decisions re-run the
submitted model/notebook against a separate private word list.

250,000 words, one game at a time (not batched) -- prints progress every
20,000 words with an ETA. Expect roughly the same ~46 minutes the other
recurrent branches took.

In [ ]:
!python src/generate_submission_gru.py

## Save outputs

Anything under `/kaggle/working/` is downloadable from the notebook's
Output tab after the run finishes.

In [ ]:
import shutil
shutil.copy("src/gru_attn_masker.pt", "/kaggle/working/gru_attn_masker.pt")
shutil.copy("submission.csv", "/kaggle/working/submission.csv")
print("saved gru_attn_masker.pt and submission.csv to /kaggle/working/ -- download from the Output tab")